# Notebook 03: Model Training & Comparison

**AI Power Electronics Diagnostics — IEEE IES Industrial AI Lab**

This notebook trains all 5 models on the synthetic inverter fault dataset
and compares their performance:

| Model | Input | Architecture |
|---|---|---|
| **1D CNN** | Raw waveform (C, T) | Residual Conv1D blocks |
| **Spectrogram CNN** | STFT image (C, H, W) | ResNet-18 on spectrograms |
| **Transformer** | Patches (C, T) | Multi-head self-attention |
| **BiLSTM** | Sequence (C, T) | Bidirectional LSTM + attention |
| **Autoencoder** | Raw waveform (C, T) | Unsupervised reconstruction |

> **Note:** For a full training run use `python training/train.py --model <name>`. This notebook runs a quick training demo with fewer epochs for illustration.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

from datasets.synthetic import InverterFaultSimulator
from datasets.loaders.base_loader import BaseDatasetLoader
from models import MODEL_REGISTRY
from training.utils import make_dataloaders, get_device, AverageMeter

device = get_device('auto')
print(f'Device: {device}')

# Dataset
sim = InverterFaultSimulator()
X, y = sim.generate_dataset(n_per_class=150, window_size=1024)
print(f'Dataset: X={X.shape}, classes={len(np.unique(y))}')

In [ ]:
# Train/val/test split
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.176, stratify=y_train_val, random_state=42)

train_loader, val_loader, test_loader = make_dataloaders(
    X_train, y_train, X_val, y_val, X_test, y_test,
    batch_size=64, num_workers=0
)
print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')

In [ ]:
def quick_train(model_name, epochs=15, n_classes=9, n_channels=6):
    """Quick training loop for notebook demo."""
    import torch.nn as nn
    
    model = MODEL_REGISTRY[model_name](
        n_channels=n_channels,
        n_classes=n_classes if model_name != 'autoencoder' else None,
        window_size=1024 if model_name != 'autoencoder' else 1024,
    ).to(device)
    
    # Skip autoencoder in supervised comparison
    if model_name == 'autoencoder':
        return model, []
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_losses = []
    for epoch in range(epochs):
        model.train()
        loss_sum = 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item()
        scheduler.step()
        train_losses.append(loss_sum / len(train_loader))
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/{epochs}  loss={train_losses[-1]:.4f}')
    
    return model, train_losses

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            logits = model(X_b.to(device))
            preds.extend(logits.argmax(-1).cpu().numpy())
            labels.extend(y_b.numpy())
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return acc, f1

print('Training functions ready.')

In [ ]:
results = {}
supervised_models = ['cnn_waveform', 'transformer', 'bilstm']

for model_name in supervised_models:
    print(f'\nTraining {model_name}...')
    model, losses = quick_train(model_name, epochs=20)
    acc, f1 = evaluate(model, test_loader)
    results[model_name] = {'accuracy': acc, 'macro_f1': f1, 'params': model.count_parameters()}
    print(f'  → Test Acc: {acc*100:.1f}%  Macro-F1: {f1:.4f}  Params: {model.count_parameters():,}')

In [ ]:
df = pd.DataFrame(results).T
df['accuracy_pct'] = df['accuracy'] * 100
print('\n=== Results Summary ===')
print(df[['accuracy_pct', 'macro_f1', 'params']].to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['accuracy_pct'].plot(kind='bar', ax=axes[0], color='#1f77b4', edgecolor='k')
axes[0].set_title('Test Accuracy (%)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 105)
axes[0].tick_params(axis='x', rotation=30)

df['macro_f1'].plot(kind='bar', ax=axes[1], color='#2ca02c', edgecolor='k')
axes[1].set_title('Macro F1 Score')
axes[1].set_ylabel('F1')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()
print('Notebook 03 complete.')